# 📏 Notebook 6 — Evaluation Framework (Phase 3)

هذا الـ notebook يشرح ويُشغّل إطار التقييم:
1. **Retrieval metrics** — Precision@K, Recall@K, MRR, Hit Rate
2. **Generation metrics** — BLEU, ROUGE-1/2/L
3. **Performance** — Latency (avg, p95), Throughput
4. تحليل النتائج بالرسوم

> ⚠️ يحتاج: Qdrant شغّال + collection مبنية + GEMINI_API_KEY
---

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from app.config import get_settings
from app.rag.pipeline import RAGPipeline
from app.evaluation.evaluator import (
    Evaluator, precision_at_k, recall_at_k, reciprocal_rank,
    bleu_score, rouge_scores, EvaluationReport
)
from app.rag.preprocessing import clean_text

settings = get_settings()
print('Settings loaded.')
print(f'Dataset: {settings.resolve(settings.dataset_path)}')
print(f'Top-K: {settings.top_k}')

## 1. Retrieval Metrics — شرح بمثال يدوي

In [ ]:
# مثال بسيط لفهم كل مقياس
print('═'*60)
print('سيناريو:')
print('  gold_row = 42  (الصف الصحيح في الـ dataset)')
print('  retrieved = [7, 42, 15, 99, 3]  (ترتيب النتائج)')
print('  top_k = 5')
print('═'*60)

gold     = {42}
retrieved = [7, 42, 15, 99, 3]
k = 5

p = precision_at_k(retrieved, gold, k)
r = recall_at_k(retrieved, gold, k)
rr = reciprocal_rank(retrieved, gold)

print(f'\nPrecision@{k} = {p:.4f}')
print(f'  → من {k} نتائج، 1 صحيحة → 1/{k} = {1/k:.2f}')
print(f'\nRecall@{k} = {r:.4f}')
print(f'  → من النتائج الصحيحة المتوقعة (1 فقط)، وجدنا 1 → 100%')
print(f'\nReciprocal Rank = {rr:.4f}')
print(f'  → الإجابة ظهرت في المرتبة 2 → 1/2 = 0.5')
print()

# سيناريو آخر — ما وجدناش الإجابة
print('سيناريو آخر: لم تُسترجع الإجابة الصحيحة')
retrieved_miss = [7, 15, 99, 3, 88]
p2  = precision_at_k(retrieved_miss, gold, k)
r2  = recall_at_k(retrieved_miss, gold, k)
rr2 = reciprocal_rank(retrieved_miss, gold)
print(f'Precision@5={p2:.2f} | Recall@5={r2:.2f} | RR={rr2:.2f}  ← كلها صفر')

## 2. Generation Metrics — BLEU و ROUGE

In [ ]:
reference = "Tom Brady holds the record for most wins with 220, playing for the New England Patriots."

candidates = {
    'Perfect match':   reference,
    'Good answer':     "Tom Brady holds the NFL record for most wins with 220 total victories.",
    'Partial':         "Tom Brady has the most wins in football history.",
    'Wrong answer':    "Patrick Mahomes is considered the best quarterback today.",
    'Empty':           "",
}

print(f'Reference: {reference}')
print()
print(f'{"Candidate":<25} {"BLEU":>6} {"R1":>6} {"R2":>6} {"RL":>6}')
print('─' * 55)
for name, cand in candidates.items():
    b  = bleu_score(reference, cand)
    rs = rouge_scores(reference, cand)
    print(f'{name:<25} {b:>6.3f} {rs["rouge1"]:>6.3f} {rs["rouge2"]:>6.3f} {rs["rougeL"]:>6.3f}')

In [ ]:
# شرح الفرق بين BLEU و ROUGE
print('''
BLEU (Bilingual Evaluation Understudy):
  → يقيس دقة n-grams (أزواج كلمات) في الإجابة مقارنة بالمرجع
  → يُعاقب على الإجابات القصيرة (brevity penalty)
  → range: 0.0 → 1.0

ROUGE-1: تطابق الكلمات المنفردة (unigrams)
ROUGE-2: تطابق أزواج الكلمات (bigrams) — أصعب
ROUGE-L: أطول تسلسل مشترك (subsequence) — يعكس الترتيب

في RAG، الإجابة المولّدة قد تكون صحيحة لكن بكلمات مختلفة
عن المرجع — BLEU/ROUGE تُعطي صورة تقريبية وليست مطلقة.
''')

## 3. تشغيل Evaluator الكامل

In [ ]:
# ⚠️ هذا سيستهلك API calls لـ Gemini
# ابدأ بعدد صغير (5-10) للاختبار

NUM_SAMPLES = 5   # ← زوّد للحصول على نتائج أدق (max 200)

print(f'Running evaluation on {NUM_SAMPLES} samples...')
pipeline  = RAGPipeline(settings)
evaluator = Evaluator(pipeline)

report = evaluator.run(
    num_samples=NUM_SAMPLES,
    top_k=settings.top_k,
    include_generation=True
)

print('\n✅ Evaluation complete!')
print(json.dumps({
    'retrieval':   report.retrieval,
    'generation':  report.generation,
    'performance': report.performance,
}, indent=2))

In [ ]:
# عرض تفاصيل العينات
print(f'\nSample Details ({min(5, len(report.samples))} of {report.num_samples}):')
print(f'{"Question":<50} {"Hit":<5} {"Conf":<6}')
print('─' * 65)
for s in report.samples[:5]:
    hit_mark = '✓' if s['hit'] else '✗'
    print(f'{s["question"][:48]:<50} {hit_mark:<5} {s["confidence"]:.3f}')

## 4. تصوير النتائج

In [ ]:
fig = plt.figure(figsize=(16, 10), facecolor='#0f1117')
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

COLORS = ['#6366f1', '#22d3ee', '#f59e0b', '#22c55e', '#ef4444']

# ── Retrieval metrics bar
ax1 = fig.add_subplot(gs[0, :2])
ax1.set_facecolor('#1a1d27')
ret = report.retrieval
names  = list(ret.keys())
values = list(ret.values())
bars = ax1.bar(names, values, color=COLORS[:len(names)], edgecolor='none')
ax1.bar_label(bars, fmt='%.3f', color='white', fontsize=10, padding=3)
ax1.set_ylim(0, 1.15)
ax1.set_title('Retrieval Metrics', color='white', fontsize=12)
ax1.tick_params(colors='#b0b0b0')
ax1.grid(axis='y', alpha=0.3)

# ── Generation metrics bar
ax2 = fig.add_subplot(gs[0, 2])
ax2.set_facecolor('#1a1d27')
gen = report.generation
if gen:
    gnames  = list(gen.keys())
    gvalues = list(gen.values())
    bars2 = ax2.bar(gnames, gvalues, color=COLORS[:len(gnames)], edgecolor='none')
    ax2.bar_label(bars2, fmt='%.3f', color='white', fontsize=9, padding=3)
    ax2.set_ylim(0, 1.15)
    ax2.set_title('Generation Metrics', color='white', fontsize=12)
    ax2.tick_params(axis='x', rotation=30, colors='#b0b0b0')
    ax2.tick_params(axis='y', colors='#b0b0b0')
    ax2.grid(axis='y', alpha=0.3)

# ── Performance metrics
ax3 = fig.add_subplot(gs[1, :2])
ax3.set_facecolor('#1a1d27')
perf = report.performance
perf_display = {k: v for k, v in perf.items() if 'ms' in k or 'qps' in k}
if perf_display:
    pnames  = list(perf_display.keys())
    pvalues = list(perf_display.values())
    bars3 = ax3.bar(pnames, pvalues, color='#22c55e', edgecolor='none')
    ax3.bar_label(bars3, fmt='%.1f', color='white', fontsize=9, padding=3)
    ax3.set_title('Performance (latency ms / throughput qps)', color='white', fontsize=11)
    ax3.tick_params(axis='x', rotation=20, colors='#b0b0b0')
    ax3.tick_params(axis='y', colors='#b0b0b0')
    ax3.grid(axis='y', alpha=0.3)

# ── Hit/Miss pie
ax4 = fig.add_subplot(gs[1, 2])
ax4.set_facecolor('#1a1d27')
hit_rate = report.retrieval.get('hit_rate', 0)
ax4.pie([hit_rate, 1-hit_rate],
        labels=[f'Hit\n{hit_rate:.1%}', f'Miss\n{1-hit_rate:.1%}'],
        colors=['#22c55e','#ef4444'],
        autopct='%1.0f%%', startangle=90,
        textprops={'color':'white', 'fontsize':11})
ax4.set_title('Hit Rate', color='white', fontsize=12)

plt.suptitle(f'Evaluation Report — {report.num_samples} samples, top_k={report.top_k}',
             color='white', fontsize=14, y=1.01)
plt.show()

## 5. تحليل عينة: ماذا كانت الأسئلة الصحيحة والخاطئة؟

In [ ]:
hits   = [s for s in report.samples if s.get('hit')]
misses = [s for s in report.samples if not s.get('hit')]

print(f'✅ Hit ({len(hits)}): أسئلة أصابت الإجابة الصحيحة:')
for s in hits[:3]:
    print(f'   Q: {s["question"][:65]}')
    print(f'   A: {s["answer"][:80]}...')
    print(f'   conf={s["confidence"]:.3f}')
    print()

print(f'\n❌ Miss ({len(misses)}): أسئلة لم تُصَب:')
for s in misses[:3]:
    print(f'   Q: {s["question"][:65]}')
    print(f'   Retrieved rows: {s["retrieved_rows"]} | Gold row: {s["gold_row"]}')
    print(f'   conf={s["confidence"]:.3f}')
    print()

In [ ]:
# ملخص التقرير
print('═' * 60)
print(f'  📊 Evaluation Summary — {report.num_samples} samples')
print('═' * 60)
print(f'  Retrieval:')
for k, v in report.retrieval.items():
    print(f'    {k:<25}: {v:.4f}')
if report.generation:
    print(f'  Generation:')
    for k, v in report.generation.items():
        print(f'    {k:<25}: {v:.4f}')
print(f'  Performance:')
for k, v in report.performance.items():
    print(f'    {k:<25}: {v}')
print('═' * 60)